In [0]:

from pyspark.sql import functions as F

# DSAI3202 — Flight Delay Prediction
# Phase 2: Prepare model training data

schema = "dsai3202_databricks.proj_flight60106354"

source_table = f"{schema}.flights_features_2025_v2"

df = spark.table(source_table)

print("PHASE 2 — DATA LOADING")
print("Source:", source_table)
print("Total records:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()
display(df.limit(10))

assert df.count() == 6879484
assert "ArrDel15" in df.columns
assert "ArrDelay" not in df.columns

print("PASS: Phase 2 dataset loaded successfully")
print("PASS: Prediction target available")
print("PASS: No actual arrival-delay leakage")


PHASE 2 — DATA LOADING
Source: dsai3202_databricks.proj_flight60106354.flights_features_2025_v2
Total records: 6879484
Total columns: 13
root
 |-- FlightDate: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: integer (nullable = true)
 |-- Month: integer (nullable = true)



FlightDate,Year,Reporting_Airline,Origin,Dest,CRSDepTime,DepHour,DayOfWeek,IsWeekend,Quarter,Route,ArrDel15,Month
2025-10-10,2025,AA,SAN,PHX,615,6,6,0,4,SAN-PHX,0,10
2025-10-11,2025,AA,SAN,PHX,615,6,7,1,4,SAN-PHX,0,10
2025-10-12,2025,AA,SAN,PHX,615,6,1,1,4,SAN-PHX,0,10
2025-10-13,2025,AA,SAN,PHX,615,6,2,0,4,SAN-PHX,0,10
2025-10-14,2025,AA,SAN,PHX,615,6,3,0,4,SAN-PHX,0,10
2025-10-15,2025,AA,SAN,PHX,615,6,4,0,4,SAN-PHX,0,10
2025-10-16,2025,AA,SAN,PHX,615,6,5,0,4,SAN-PHX,0,10
2025-10-17,2025,AA,SAN,PHX,615,6,6,0,4,SAN-PHX,0,10
2025-10-18,2025,AA,SAN,PHX,615,6,7,1,4,SAN-PHX,0,10
2025-10-19,2025,AA,SAN,PHX,615,6,1,1,4,SAN-PHX,0,10


PASS: Phase 2 dataset loaded successfully
PASS: Prediction target available
PASS: No actual arrival-delay leakage


In [0]:

# Chronological splitting prevents random mixing
# of older and future flight records.

train_df = df.filter(
    F.col("FlightDate") < F.lit("2025-09-01").cast("date")
)

val_df = df.filter(
    (F.col("FlightDate") >= F.lit("2025-09-01").cast("date"))
    & (F.col("FlightDate") < F.lit("2025-11-01").cast("date"))
)

test_df = df.filter(
    F.col("FlightDate") >= F.lit("2025-11-01").cast("date")
)

train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()

print("PHASE 2 — CHRONOLOGICAL SPLIT")
print("Training records:", train_count)
print("Validation records:", val_count)
print("Testing records:", test_count)
print("Total:", train_count + val_count + test_count)

assert train_count > 0
assert val_count > 0
assert test_count > 0

assert train_count + val_count + test_count == 6879484

print("PASS: All flights assigned to one split")
print("PASS: Training, validation and test periods do not overlap")


PHASE 2 — CHRONOLOGICAL SPLIT
Training records: 4592366
Validation records: 1159898
Testing records: 1127220
Total: 6879484
PASS: All flights assigned to one split
PASS: Training, validation and test periods do not overlap


In [0]:

# Compare class balance across chronological splits.

split_dfs = [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]

results = []

for name, dataset in split_dfs:
    stats = dataset.agg(
        F.count("*").alias("Total"),
        F.sum("ArrDel15").alias("Delayed")
    ).first()

    total = int(stats["Total"])
    delayed = int(stats["Delayed"])
    rate = round(100 * delayed / total, 2)

    results.append((name, total, delayed, rate))

summary_df = spark.createDataFrame(
    results,
    ["Split", "TotalFlights", "DelayedFlights", "DelayRatePercent"]
)

display(summary_df)

print("PASS: Class distribution analysis completed")
print("PASS: Dataset ready for baseline modeling")


Split,TotalFlights,DelayedFlights,DelayRatePercent
Training,4592366,1052245,22.91
Validation,1159898,215110,18.55
Testing,1127220,267283,23.71


PASS: Class distribution analysis completed
PASS: Dataset ready for baseline modeling


In [0]:

# PHASE 2 — Create fixed, versioned data snapshots
# Run this publication cell only once.

storage = "stflight60106354"
snapshot_version = "phase2_20251118_v1"

snapshot_parent = (
    f"abfss://curated@{storage}.dfs.core.windows.net/"
    "flight_delays/phase2/"
)

snapshot_root = snapshot_parent + snapshot_version + "/"

# Create the parent directory, then prevent accidental overwrite.
dbutils.fs.mkdirs(snapshot_parent)

existing = [x.name.rstrip("/") for x in dbutils.fs.ls(snapshot_parent)]

if snapshot_version in existing:
    raise ValueError(
        "Snapshot already exists. Do not overwrite it. "
        "Verify the existing snapshot instead."
    )

paths = {
    "train": snapshot_root + "train/",
    "validation": snapshot_root + "validation/",
    "test": snapshot_root + "test/"
}

# Preserve the original feature columns and exact split membership.
for name, dataset in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    (
        dataset.write
        .mode("overwrite")
        .format("parquet")
        .partitionBy("Month")
        .save(paths[name])
    )
    print("Saved:", name, paths[name])

print("PASS: Phase 2 snapshot files created")


Saved: train abfss://curated@stflight60106354.dfs.core.windows.net/flight_delays/phase2/phase2_20251118_v1/train/
Saved: validation abfss://curated@stflight60106354.dfs.core.windows.net/flight_delays/phase2/phase2_20251118_v1/validation/
Saved: test abfss://curated@stflight60106354.dfs.core.windows.net/flight_delays/phase2/phase2_20251118_v1/test/
PASS: Phase 2 snapshot files created


In [0]:

# Read the saved snapshots instead of relying on notebook variables.

train_saved = spark.read.parquet(paths["train"])
val_saved = spark.read.parquet(paths["validation"])
test_saved = spark.read.parquet(paths["test"])

saved_counts = {
    "Training": train_saved.count(),
    "Validation": val_saved.count(),
    "Testing": test_saved.count()
}

for name, count in saved_counts.items():
    print(f"{name}: {count:,} records")

assert saved_counts["Training"] == train_count
assert saved_counts["Validation"] == val_count
assert saved_counts["Testing"] == test_count

assert sum(saved_counts.values()) == 6879484

# Confirm correct time boundaries.
assert train_saved.filter(
    F.col("FlightDate") >= F.lit("2025-09-01").cast("date")
).count() == 0

assert val_saved.filter(
    (F.col("FlightDate") < F.lit("2025-09-01").cast("date")) |
    (F.col("FlightDate") >= F.lit("2025-11-01").cast("date"))
).count() == 0

assert test_saved.filter(
    F.col("FlightDate") < F.lit("2025-11-01").cast("date")
).count() == 0

print("Snapshot version:", snapshot_version)
print("PASS: All saved datasets verified")
print("PASS: Chronological boundaries verified")
print("PASS: Training data is ready for modeling")


Training: 4,592,366 records
Validation: 1,159,898 records
Testing: 1,127,220 records
Snapshot version: phase2_20251118_v1
PASS: All saved datasets verified
PASS: Chronological boundaries verified
PASS: Training data is ready for modeling
